# 01 — Data Acquisition

## Retention Intelligence Platform

### Objective

Acquire the primary IBM Telco Customer Churn dataset and establish a reproducible, traceable raw data artifact.

This notebook is responsible for:

1. Recording the authoritative source.
2. Downloading the raw dataset.
3. Recording acquisition metadata.
4. Verifying file integrity using SHA-256.
5. Confirming the raw artifact exists at the expected location.

### Important rule

The raw dataset is immutable.

No cleaning, transformation, filtering, feature engineering, or EDA is performed in this notebook.

## Imports and configuration

In [5]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import urllib.request

PROJECT_ROOT = Path("../retention-intelligence")

RAW_DIR = PROJECT_ROOT / "data/raw/telco"
RAW_FILE = RAW_DIR / "Telco-Customer-Churn.csv"

SOURCE_URL = (
    "https://raw.githubusercontent.com/"
    "IBM/telco-customer-churn-on-icp4d/"
    "master/data/Telco-Customer-Churn.csv"
)

SOURCE_REPOSITORY = (
    "https://github.com/IBM/telco-customer-churn-on-icp4d"
)

EXPECTED_SHA256 = (
    "16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91"
)

RAW_DIR.mkdir(parents=True, exist_ok=True)

## Acquire the dataset

In [6]:
if RAW_FILE.exists():
    print(f"Raw file already exists: {RAW_FILE}")
else:
    print("Downloading dataset...")
    urllib.request.urlretrieve(SOURCE_URL, RAW_FILE)
    print(f"Downloaded: {RAW_FILE}")

Downloaded: ../retention-intelligence/data/raw/telco/Telco-Customer-Churn.csv


## Calculate SHA-256

In [ ]:
def calculate_sha256(path: Path) -> str:
    sha256 = hashlib.sha256()

    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            sha256.update(chunk)

    return sha256.hexdigest()


actual_sha256 = calculate_sha256(RAW_FILE)

print("Expected:")
print(EXPECTED_SHA256)

print("\nActual:")
print(actual_sha256)

print("\nMatch:", actual_sha256 == EXPECTED_SHA256)

## Acquisition metadata

In [8]:
from datetime import datetime, timezone
import hashlib

# 1. Calculate the SHA-256 hash of the raw file
sha256_hash = hashlib.sha256()
with open(RAW_FILE, "rb") as f:
    for byte_block in iter(lambda: f.read(4096), b""):
        sha256_hash.update(byte_block)
actual_sha256 = sha256_hash.hexdigest()

# 2. Build your acquisition metadata dictionary
acquisition_metadata = {
    "dataset": "IBM Telco Customer Churn",
    "source_organization": "IBM",
    "source_repository": SOURCE_REPOSITORY,
    "source_file": SOURCE_URL,
    "local_path": str(RAW_FILE.relative_to(PROJECT_ROOT)),
    "acquisition_timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "file_name": RAW_FILE.name,
    "file_size_bytes": RAW_FILE.stat().st_size,
    "sha256": actual_sha256,
}

acquisition_metadata

{'dataset': 'IBM Telco Customer Churn',
 'source_organization': 'IBM',
 'source_repository': 'https://github.com/IBM/telco-customer-churn-on-icp4d',
 'source_file': 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv',
 'local_path': 'data/raw/telco/Telco-Customer-Churn.csv',
 'acquisition_timestamp_utc': '2026-10-05T22:49:42.108583+00:00',
 'file_name': 'Telco-Customer-Churn.csv',
 'file_size_bytes': 970457,
 'sha256': '16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91'}

## Save machine-readable provenance

In [ ]:
METADATA_FILE = RAW_DIR / "acquisition_metadata.json"

with METADATA_FILE.open("w") as f:
    json.dump(acquisition_metadata, f, indent=2)

print(f"Saved: {METADATA_FILE}")

## Basic artifact verification

In [9]:
assert RAW_FILE.exists(), "Raw dataset does not exist."
assert RAW_FILE.stat().st_size > 0, "Raw dataset is empty."
assert actual_sha256 == EXPECTED_SHA256, "SHA-256 mismatch."

print("✓ Raw artifact verified successfully.")

✓ Raw artifact verified successfully.


## Human-readable summary

In [10]:
print("DATA ACQUISITION COMPLETE")
print("=" * 40)
print(f"Dataset:       {acquisition_metadata['dataset']}")
print(f"Source:        {acquisition_metadata['source_organization']}")
print(f"Local file:    {acquisition_metadata['local_path']}")
print(f"Size:          {acquisition_metadata['file_size_bytes']:,} bytes")
print(f"SHA-256:       {acquisition_metadata['sha256']}")
print(f"Integrity:     {'PASS' if actual_sha256 == EXPECTED_SHA256 else 'FAIL'}")

DATA ACQUISITION COMPLETE
Dataset:       IBM Telco Customer Churn
Source:        IBM
Local file:    data/raw/telco/Telco-Customer-Churn.csv
Size:          970,457 bytes
SHA-256:       16320c9c1ec72448db59aa0a26a0b95401046bef5d02fd3aeb906448e3055e91
Integrity:     PASS
